# CS2227: Artificial Intelligence and Machine Learning
## Lab 4: Handling Imbalanced Datasets using SMOTE

**Objective:** Understand the Accuracy Paradox and use SMOTE (Synthetic Minority Oversampling Technique) to balance an imbalanced dataset by interpolating new minority class samples.

---
### What is the Accuracy Paradox?
If 99% of data belongs to Class 0 and 1% to Class 1, a model that always predicts Class 0 gets 99% accuracy — but is completely useless for detecting Class 1.

### How does SMOTE work?
Instead of duplicating minority samples, SMOTE **synthesizes** new points between existing minority points using interpolation:
```
New Point = A + λ × (B - A)
```
Where A and B are minority points, and λ is a random number between 0 and 1.

---
## Step 1: Install Required Library
We need `imbalanced-learn` for SMOTE. Run this once in your environment.

In [ ]:
# Install imbalanced-learn (only needed once)
!pip install imbalanced-learn -q

---
## Step 2: Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from collections import Counter
from imblearn.over_sampling import SMOTE

print("All libraries imported successfully!")

---
## Step 3: Generate an Imbalanced Dataset

We simulate a real-world scenario (e.g., rare disease diagnosis) where:
- **Class 0** = 90% of data (majority — healthy patients)
- **Class 1** = 10% of data (minority — diseased patients)

> `make_classification` from sklearn lets us generate synthetic datasets with controlled imbalance.

In [ ]:
# Generate a highly imbalanced dataset (90% Class 0, 10% Class 1)
X, y = make_classification(
    n_samples=1000,           # Total data points
    n_features=2,             # 2 features so we can visualize in 2D
    n_redundant=0,            # No redundant features
    n_clusters_per_class=1,   # One cluster per class
    weights=[0.90],           # 90% Class 0, 10% Class 1
    random_state=42           # Reproducibility
)

# Check the class distribution
print(f"Original class distribution: {Counter(y)}")
print(f"Class 0 (Majority): {Counter(y)[0]} samples")
print(f"Class 1 (Minority): {Counter(y)[1]} samples")

---
## Step 4: Apply SMOTE

SMOTE synthesizes new minority class points by interpolating between existing ones.

**Key parameters:**
- `sampling_strategy='minority'` → Balance until minority equals majority
- `k_neighbors=5` → Uses 5 nearest minority neighbors to create each new point

> ⚠️ **Important:** In real projects, always split your data FIRST (train/test), then apply SMOTE **only** on the training set. Applying SMOTE before splitting causes **data leakage**.

In [ ]:
# Initialize SMOTE
smote = SMOTE(
    sampling_strategy='minority',  # Oversample until minority = majority count
    random_state=42,               # Reproducibility
    k_neighbors=5                  # Number of nearest neighbors to use
)

# Fit and resample the data
X_resampled, y_resampled = smote.fit_resample(X, y)

# Check the new distribution
print(f"Resampled class distribution: {Counter(y_resampled)}")
print(f"Class 0 (Majority): {Counter(y_resampled)[0]} samples")
print(f"Class 1 (Minority — now balanced): {Counter(y_resampled)[1]} samples")

---
## Step 5: Visualize Before vs After SMOTE

We plot both datasets side by side so you can see the effect of SMOTE on the feature space.
- **Red points** = Class 0 (Majority)
- **Blue points** = Class 1 (Minority)

After SMOTE, you should see more blue points — these are the **interpolated synthetic samples**.

In [ ]:
# Set up a side-by-side comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# --- Plot 1: Original Imbalanced Data ---
ax1.scatter(X[:, 0], X[:, 1], c=y, alpha=0.6, cmap='coolwarm', edgecolors='k')
ax1.set_title("Original Imbalanced Data\n(Accuracy Paradox Risk)")
ax1.set_xlabel("Feature 1")
ax1.set_ylabel("Feature 2")

# --- Plot 2: Balanced Data after SMOTE ---
ax2.scatter(X_resampled[:, 0], X_resampled[:, 1], c=y_resampled, alpha=0.6,
            cmap='coolwarm', edgecolors='k')
ax2.set_title("Balanced Data after SMOTE\n(Interpolated Minority Points)")
ax2.set_xlabel("Feature 1")
ax2.set_ylabel("Feature 2")

plt.tight_layout()
plt.show()

print("\nObservation: The right plot shows many more minority (blue) points — these are SMOTE-generated synthetic samples.")

---
## Summary

| Metric | Before SMOTE | After SMOTE |
|---|---|---|
| Class 0 Count | ~900 | ~900 |
| Class 1 Count | ~100 | ~900 |
| Balanced? | ❌ No | ✅ Yes |

**Key Takeaways:**
1. SMOTE generates new points using **interpolation**, not duplication — reducing overfitting risk.
2. Always apply SMOTE **after** the train/test split to avoid data leakage.
3. Use metrics like **Precision, Recall, F1-Score** instead of Accuracy on imbalanced data.